# Spotify Genre Classification

In [ ]:
cd '/Users/wiseer/Documents/github/listen-wiseer/src/'

In [ ]:
import pandas as pd
from analysis.data import *
from analysis.genre import *
from analysis.plotting import *

from models.clustering import *
from utils.const import *

data = LoadPlaylistData()

In [ ]:
# TODO: allow input for genre or some other filtering method

In [ ]:
def return_filtered_recommendations():
    df = data.return_enoa_data()
    df = df[df.gen_8 == "r&b"]
    numeric_features = num_features  # let's try this out for genre
    categorical_features = ["time_signature", "key_mode", "decade"]
    # label encoding for decade; ordinal for time_signature and key_mode

    X = df[numeric_features + categorical_features]
    one_hot_encoded = pd.get_dummies(X[categorical_features])
    X = pd.concat([X, one_hot_encoded], axis=1)
    X = X.drop(categorical_features, axis=1)
    return X

In [ ]:
n_components = select_pca_n_components(X)
# TODO: save output

In [ ]:
n_clusters = plot_elbow_method(X)
# TODO: save output

In [ ]:
plot_sihloette_scores(X)
# TODO: save output

## Run Model

In [ ]:
def config_model_pipeline(n_components, n_clusters):
    pipelines = []

    # KMeans
    kmeans_pipeline = Pipeline(
        [
            ("scaler", MinMaxScaler()),
            ("ordinal_encoder", OrdinalEncoder()),
            ("pca", PCA(n_components=n_components)),
            ("classifier", KMeans(n_clusters=n_clusters, n_init=10)),
        ]
    )
    pipelines.append(kmeans_pipeline)

    # SpectralClustering
    spectral_pipeline = Pipeline(
        [
            ("scaler", MinMaxScaler()),
            # ("ordinal_encoder", OrdinalEncoder()),
            # ("pca", PCA(n_components=n_components)),
            ("classifier", SpectralClustering(n_clusters=n_clusters)),
        ]
    )
    pipelines.append(spectral_pipeline)

    # AgglomerativeClustering
    agglomerative_pipeline = Pipeline(
        [
            ("scaler", MinMaxScaler()),
            # ("ordinal_encoder", OrdinalEncoder()),
            # ("pca", PCA(n_components=n_components)),
            ("classifier", AgglomerativeClustering(n_clusters=n_clusters)),
        ]
    )
    pipelines.append(agglomerative_pipeline)

    # DBSCAN
    dbscan_pipeline = Pipeline(
        [
            ("scaler", MinMaxScaler()),
            # ("ordinal_encoder", OrdinalEncoder()),
            # ("pca", PCA(n_components=n_components)),
            ("classifier", DBSCAN()),
        ]
    )
    pipelines.append(dbscan_pipeline)

    return pipelines

In [ ]:
## Dunn Index
# def dunn_index(X, labels):
#    n_clusters = len(np.unique(labels))
#    min_intercluster_distances = np.min(pairwise_distances(X, metric="euclidean"))
#    max_intracluster_distances = 0
#
#    for i in range(n_clusters):
#        cluster_points = X[labels == i]
#        intracluster_distances = pairwise_distances(cluster_points, metric="euclidean")
#        max_intracluster_distances = max(
#            np.max(intracluster_distances), max_intracluster_distances
#        )
#
#    dunn = min_intercluster_distances / max_intracluster_distances
#    return dunn

In [ ]:
from sklearn.metrics import davies_bouldin_score

pipelines = config_model_pipeline(n_components, n_clusters)

# return clusters
for model in pipelines:
    model_name = type(model.named_steps["classifier"]).__name__
    labels = model.fit_predict(X).tolist()
    df[model_name] = labels

    # return metrics for models
    silhouette_avg = silhouette_score(X, labels)
    print(model_name + f" Silhouette Score: {silhouette_avg}")
    ch_score = calinski_harabasz_score(X, labels)
    print(model_name + f" Calinski-Harabasz Index: {ch_score}")

    db_score = davies_bouldin_score(X, labels)
    print(model_name + f" Davies-Bouldin Index: {db_score}")

    # Inertia (within-cluster sum of squares) for k-means
    if "KMeans" in model_name:
        inertia = model.named_steps["classifier"].inertia_
        print(model_name + f" Inertia: {inertia}")

    # compare cluster features
    plot_tsne(X, n_components, labels, model_name)
    plot_cluster_features(df, model_name)


# TODO: add function to select model with highest ranking between these two scores
# calinski measures variance within clusters: variance between clusters
# silhoette indicates similarity objects within a cluster are

In [ ]:
# df[df.KMeans == 0]['genres'].value_counts().plot(kind='bar')
df.SpectralClustering.value_counts().plot(kind="bar")

In [ ]:
df[df.SpectralClustering == 0][["id", "track_name", "artist_names", "genres"]]

In [ ]:
df[df.SpectralClustering == 1][["id", "track_name", "artist_names", "genres"]]

In [ ]:
# TODO: once we have track_ids, create new playlist and append as uris